# Milan Cross-Comparison Start
This notebook sets up a first cross-comparison between Airbnb activity and long-term rental levels in Milan. The goal is to build a clean base table and produce one generic visualization to guide deeper analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## Load Datasets
We use three sources: cleaned Airbnb listings, Airbnb reviews with dates, and OMI rental quotations.

In [ ]:
listings = pd.read_csv('Data/listings_milan_clean.csv')
reviews = pd.read_csv('Data/reviews_milan.csv')
rentals = pd.read_csv('Data/total_rentals.csv', sep=';')

print('Listings shape:', listings.shape)
print('Reviews shape:', reviews.shape)
print('Rentals shape:', rentals.shape)

## Build Comparable Yearly Indicators
We create yearly Airbnb activity from reviews and yearly rental level from OMI values, then merge on Year.

In [ ]:
reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')
reviews['Year'] = reviews['date'].dt.year

airbnb_yearly = (
    reviews.dropna(subset=['Year'])
    .groupby('Year')
    .agg(review_count=('listing_id', 'size'), active_listings=('listing_id', 'nunique'))
    .reset_index()
)

for col in ['Loc_min', 'Loc_max']:
    rentals[col] = pd.to_numeric(
        rentals[col].astype(str).str.replace(',', '.', regex=False),
        errors='coerce'
    )

rentals['Year'] = pd.to_numeric(rentals['Year'], errors='coerce')
rentals['rent_mid'] = (rentals['Loc_min'] + rentals['Loc_max']) / 2

rentals_yearly = (
    rentals.dropna(subset=['Year', 'rent_mid'])
    .groupby('Year', as_index=False)['rent_mid']
    .mean()
)

comparison = airbnb_yearly.merge(rentals_yearly, on='Year', how='inner').sort_values('Year')
comparison.head()

## Generic Visualization
The first view compares standardized yearly trends and includes a simple scatter to inspect the relationship between Airbnb activity and average rent level.

In [ ]:
viz = comparison.copy()
viz['reviews_z'] = (viz['review_count'] - viz['review_count'].mean()) / viz['review_count'].std(ddof=0)
viz['rent_z'] = (viz['rent_mid'] - viz['rent_mid'].mean()) / viz['rent_mid'].std(ddof=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(viz['Year'], viz['reviews_z'], marker='o', label='Airbnb review activity (z-score)')
axes[0].plot(viz['Year'], viz['rent_z'], marker='o', label='Average OMI rent level (z-score)')
axes[0].set_title('Standardized Yearly Trends')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Standardized value')
axes[0].legend()

sns.regplot(data=viz, x='review_count', y='rent_mid', ax=axes[1], scatter_kws={'s': 60, 'alpha': 0.85})
axes[1].set_title('Airbnb Activity vs Rent Level')
axes[1].set_xlabel('Review count (yearly)')
axes[1].set_ylabel('Average rent midpoint')

plt.tight_layout()
plt.show()